# Disney+ Content Analysis Dashboard

Interactive dashboard built with **Plotly** and **ipywidgets**. Use the filters to explore Disney+ titles by content type and release year.

In [ ]:
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, clear_output

# Load the dataset stored in the same GitHub repository
df = pd.read_csv("disney_plus_titles.csv")

# Clean the dataset
clean_df = df.copy()
clean_df["date_added"] = pd.to_datetime(clean_df["date_added"], errors="coerce")
clean_df["director"] = clean_df["director"].fillna("Unknown")
clean_df["cast"] = clean_df["cast"].fillna("Unknown")
clean_df["country"] = clean_df["country"].fillna("Unknown")
clean_df["rating"] = clean_df["rating"].fillna("Not Rated")

# Prepare dashboard fields
clean_df["duration_value"] = pd.to_numeric(
    clean_df["duration"].str.extract(r"(\\d+)")[0],
    errors="coerce"
)
clean_df["Decade"] = (clean_df["release_year"] // 10) * 10


## Interactive Dashboard

Select a content type and release-year range. The charts update automatically.

In [ ]:

import ipywidgets as widgets
from IPython.display import display, clear_output

# Prepare dashboard data
dashboard_df = clean_df.copy()

# Extract duration values
dashboard_df["duration_value"] = (
    dashboard_df["duration"]
    .str.extract(r"(\d+)")
    .astype(float)
)

# Create release decade
dashboard_df["Decade"] = (
    dashboard_df["release_year"] // 10
) * 10


# Create filters
type_filter = widgets.Dropdown(
    options=["All", "Movie", "TV Show"],
    value="All",
    description="Content Type:"
)

year_filter = widgets.IntRangeSlider(
    value=(
        int(dashboard_df["release_year"].min()),
        int(dashboard_df["release_year"].max())
    ),
    min=int(dashboard_df["release_year"].min()),
    max=int(dashboard_df["release_year"].max()),
    description="Year Range:",
    continuous_update=False
)


# Create chart output areas
out1 = widgets.Output()
out2 = widgets.Output()
out3 = widgets.Output()
out4 = widgets.Output()
out5 = widgets.Output()

total_titles = widgets.HTML()


# Dashboard function
def update_dashboard(change=None):

    # Apply filters
    filtered_df = dashboard_df[
        dashboard_df["release_year"].between(
            year_filter.value[0],
            year_filter.value[1]
        )
    ]

    if type_filter.value != "All":
        filtered_df = filtered_df[
            filtered_df["type"] == type_filter.value
        ]

    total_titles.value = (
        f"<h3>Total Titles: {len(filtered_df)}</h3>"
    )


    # Chart 1: Bar Chart
    with out1:
        clear_output(wait=True)

        content_data = (
            filtered_df["type"].value_counts().reset_index()
        )

        content_data.columns = ["Content Type", "Count"]

        fig1 = px.bar(
            content_data,
            x="Content Type",
            y="Count",
            color="Content Type",
            title="Movies vs TV Shows",
            template="plotly_dark",
            height=350
        )

        display(fig1)


    # Chart 2: Line Chart
    with out2:
        clear_output(wait=True)

        yearly_data = (
            filtered_df.groupby("release_year")
            .size()
            .reset_index(name="Count")
        )

        fig2 = px.line(
            yearly_data,
            x="release_year",
            y="Count",
            markers=True,
            title="Content Releases Over the Years",
            template="plotly_dark",
            height=350
        )

        display(fig2)


    # Prepare duration data
    if type_filter.value == "TV Show":

        duration_data = filtered_df[
            filtered_df["type"] == "TV Show"
        ].copy()

        duration_label = "Number of Seasons"

    else:

        duration_data = filtered_df[
            filtered_df["type"] == "Movie"
        ].copy()

        duration_label = "Duration (Minutes)"


    duration_data = duration_data.dropna(
        subset=["duration_value"]
    )


    # Chart 3: Scatter Plot
    with out3:
        clear_output(wait=True)

        if not duration_data.empty:

            if type_filter.value == "TV Show":
                chart_title = "TV Show Seasons vs Release Year"
            elif type_filter.value == "Movie":
                chart_title = "Movie Duration vs Release Year"
            else:
                chart_title = "Movie Duration vs Release Year (Movies Only)"

            fig3 = px.scatter(
                duration_data,
                x="release_year",
                y="duration_value",
                title=chart_title,
                labels={
                    "release_year": "Release Year",
                    "duration_value": duration_label
                },
                template="plotly_dark",
                height=350
            )

            display(fig3)


    # Chart 4: Histogram
    with out4:
        clear_output(wait=True)

        if not duration_data.empty:

            if type_filter.value == "TV Show":
                histogram_title = "Distribution of TV Show Seasons"
            elif type_filter.value == "Movie":
                histogram_title = "Distribution of Movie Durations"
            else:
                histogram_title = "Distribution of Movie Durations (Movies Only)"

            fig4 = px.histogram(
                duration_data,
                x="duration_value",
                nbins=20,
                title=histogram_title,
                labels={
                    "duration_value": duration_label
                },
                template="plotly_dark",
                height=350
            )

            display(fig4)


    # Chart 5: Heatmap
    with out5:
        clear_output(wait=True)

        heatmap_data = filtered_df.groupby(
            ["type", "Decade"]
        ).size().unstack(fill_value=0)

        if not heatmap_data.empty:

            fig5 = px.imshow(
                heatmap_data,
                text_auto=True,
                color_continuous_scale="Blues",
                title="Content Type vs Release Decade",
                template="plotly_dark",
                height=350
            )

            display(fig5)


# Connect filters
type_filter.observe(update_dashboard, names="value")
year_filter.observe(update_dashboard, names="value")


# Arrange dashboard in two columns
dashboard_layout = widgets.GridspecLayout(
    3, 2,
    height="1150px"
)

dashboard_layout[0, 0] = out1
dashboard_layout[0, 1] = out2
dashboard_layout[1, 0] = out3
dashboard_layout[1, 1] = out4
dashboard_layout[2, :] = out5


# Display dashboard
display(
    widgets.HTML(
        "<h1>Disney+ Content Analysis Dashboard</h1>"
    ),
    widgets.HBox([type_filter, year_filter]),
    total_titles,
    dashboard_layout
)

# Display initial charts
update_dashboard()
